In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

chemin_racine = os.path.abspath('..')

if chemin_racine not in sys.path:
    sys.path.append(chemin_racine)

# 6. Etude de l'explicabilité du modèle XGBoost via SHAP

Dans cette section, nous allons utiliser la bibliothèque `shap` pour pouvoir mieux comprendre les décisions prises par notre modèle XGBoost en analysant directement l'impact des variables de notre jeu de données sur les taux de risque attribués. Le but de cette démarche est d'apporter de l'explicabilité à un modèle boîte noire, chose essentielle dans le milieu bancaire où il est obligatoire de pouvoir apporter le raisonnement derrière le refus d'un crédit à un client.

## a. Explicabilité globale
Nous allons commencer par générer puis discuter les résultats du SHAP Summary Plot pour pouvoir déduire si notre modèle repose sur une logique métier saine en raccord avec l'intuition qu'un analyste de risque pourrait avoir. Ici, nous choisissons d'extraire nos SHAP values sur un échantillon représentatif du jeu de données entier, on choisit `n = 15000` soit 10% du dataset entier.

In [ ]:
import shap
from src.utils import load_pipeline
import pandas as pd
import matplotlib.pyplot as plt

raw_data_dir = "../data/raw/"
raw_train_val_file = 'cs-training.csv'

credit_data = pd.read_csv(raw_data_dir + raw_train_val_file, index_col=0, header=0)
X = credit_data.drop("SeriousDlqin2yrs", axis=1)
y = credit_data["SeriousDlqin2yrs"]

xgb_pipeline = load_pipeline("xgb_pipeline.joblib")
xgb_model = xgb_pipeline.named_steps["simple_xgb"]

preprocessor = xgb_pipeline.named_steps['preprocessor']

X_transformed = preprocessor.transform(X)
X_sample = X_transformed.sample(n=15000, random_state=67)
explainer = shap.TreeExplainer(xgb_model)

shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, show=False)
plt.savefig("../images/shap_summary_global.png", bbox_inches='tight', dpi=300)

plt.show()

<br>
<p align="center">
  <img src="../images/shap_summary_global.png" alt="SHAP Summary Plot XGBoost" width="800"/>
</p>
<br>

### Interprétation des résultats
L'analyse de ce graphique SHAP nous permet de valider la logique interne de notre modèle XGBoost et de s'assurer qu'il prend ses décisions sur des critères qui ont du sens financièrement. On tire plusieurs informations de cette distribution :
- **Le surendettement et les retards en tête :** Logiquement, les variables liées à l'utilisation des lignes de crédit (`RevolvingUtilizationOfUnsecuredLines`) et les retards de paiement (comme `NumberOfTimes90DaysLate`) dominent le classement. On observe très clairement que les valeurs élevées pour ces variables (les points rouges) étirent fortement la prédiction vers la droite, faisant exploser le risque de défaut. À l'inverse, l'absence de retards (les amas bleus) tire la prédiction vers la gauche pour sécuriser le profil.
- **L'âge, symbole de sécurité :** On constate une séparation très nette des couleurs pour la variable `age`. Les clients les plus âgés (en rouge) sont massivement concentrés à gauche du graphique, confirmant que le modèle considère l'âge comme un fort réducteur de risque. À l'inverse, être plus jeune (points bleus) vient majorer la probabilité de faire défaut.
- **Validation de la scission de la dette :** Le graphique confirme que notre traitement sur les revenus et les dettes était pertinent. Pour nos deux nouvelles variables `TrueDebtRatio` et `MonthlyDebtAmount`, on voit bien que les profils avec de fortes valeurs (en rouge) sont poussés vers la droite. Le modèle a donc bien compris qu'une charge mensuelle absolue élevée en euros ou un fort taux d'effort sont de vrais facteurs de risque.
 - **Les codes d'erreur 96 et 98 :** Notre variable `Has_System_Error_96_98` affiche un comportement très intéressant. On remarque un petit groupe de points rouges totalement isolé sur la droite (à environ +1 sur l'axe). Cela prouve que le modèle a parfaitement identifié ce flag comme un signal de risque majeur, validant notre choix de l'EDA d'avoir isolé ces profils plutôt que de les laisser biaiser nos autres statistiques.

On a donc un modèle qui a l'air très sain et cohérent avec la réalité du monde financier.